# 🇮🇩 Baseline Klasifikasi Teks Bahasa Indonesia — Siap Lomba Offline (PyTorch)

> Notebook ini dirancang sebagai **titik awal (baseline) untuk lomba tanpa internet**. Semuanya berjalan dari **folder lokal**: tidak ada unduhan model, tidak ada panggilan ke Hugging Face Hub.

| Item | Keterangan |
|---|---|
| **Tugas** | Klasifikasi teks (biner **atau** multikelas; jumlah kelas dideteksi otomatis) |
| **Bahasa** | Indonesia (termasuk teks informal/media sosial) |
| **Framework** | PyTorch + 🤗 Transformers (tanpa Keras / TensorFlow) |
| **Mode** | **Offline penuh** (`HF_HUB_OFFLINE=1`, `local_files_only=True`) |
| **Dua baseline sekaligus** | ① TF-IDF + Regresi Logistik (cepat, tanpa model) → ② IndoBERT/XLM-R fine-tuning |
| **Metrik** | Dapat dipilih: `akurasi`, `f1_makro`, `f1_biner` |

## ✅ Kenapa ada dua baseline?
Di lomba offline, risiko terbesar bukan skor kurang tinggi, melainkan **tidak punya submission sama sekali**. Karena itu:

1. **TF-IDF + Regresi Logistik** berjalan dalam hitungan detik di CPU dan langsung menghasilkan `submission_tfidf.csv` → *jaring pengaman*.
2. **Transformer** baru dijalankan sesudahnya. Jika folder model bermasalah, kamu tetap sudah punya file submission.

---

## 📑 Daftar isi
1. [Persiapan sebelum lomba (saat masih ada internet)](#1.-Persiapan-sebelum-lomba-(saat-masih-ada-internet))
2. [Setup & konfigurasi](#2.-Setup-&-konfigurasi)
3. [Memuat data](#3.-Memuat-data)
4. [Praproses teks Indonesia & pembagian data](#4.-Praproses-teks-Indonesia-&-pembagian-data)
5. [Baseline 1: TF-IDF + Regresi Logistik](#5.-Baseline-1:-TF-IDF-+-Regresi-Logistik)
6. [Baseline 2: Memuat model Transformer (offline)](#6.-Baseline-2:-Memuat-model-Transformer-(offline))
7. [DataLoader](#7.-DataLoader)
8. [Training](#8.-Training)
9. [Evaluasi](#9.-Evaluasi)
10. [Prediksi & submission](#10.-Prediksi-&-submission)
11. [Ide peningkatan, checklist hari-H & pemecahan masalah](#11.-Ide-peningkatan,-checklist-hari-H-&-pemecahan-masalah)

## 🗺️ Alur

```text
 train.csv ─► normalisasi teks ─► encode label ─► split train/val (berstrata)
                                                        │
                 ┌──────────────────────────────────────┴───────────────────────────────┐
                 ▼                                                                      ▼
   TF-IDF + Regresi Logistik                                        Tokenizer + IndoBERT/XLM-R (folder lokal)
                 │                                                                      │
        submission_tfidf.csv  ◄─ jaring pengaman            loop training → bobot terbaik → submission.csv
```

## 🔎 Cara membaca notebook
Setiap baris kode diberi komentar `#`; markdown menjelaskan konsepnya. `> 💡` tips · `> ⚠️` peringatan · `> 🧠` intuisi.

# 1. Persiapan sebelum lomba (saat masih ada internet)

Di lokasi lomba offline kamu **tidak bisa** mengunduh model. Siapkan sebelumnya:

## 1.1 Pilih model bahasa Indonesia

| Model (nama di Hugging Face) | Keunggulan | Catatan |
|---|---|---|
| `indobenchmark/indobert-base-p1` | IndoBERT dilatih di korpus Indo4B (±3,6 miliar kata); umumnya **pilihan utama** | Uncased |
| `indolem/indobert-base-uncased` | IndoBERT dari Wikipedia + berita + web (±220 juta kata) | Uncased |
| `indolem/indobertweet-base-uncased` | Khusus **Twitter/teks informal** Indonesia | Konvensi praproses khusus (lihat bagian 2) |
| `xlm-roberta-base` | Multibahasa (termasuk Indonesia); cocok jika teks **campur Inggris–Indonesia** | Perlu `sentencepiece` |
| `bert-base-multilingual-cased` | Multibahasa, ringan dan stabil | Biasanya di bawah model khusus Indonesia |

> 💡 **Aturan praktis:** teks formal/berita → `indobert-base-p1`; media sosial → `indobertweet`; campur bahasa → `xlm-roberta-base`.

## 1.2 Unduh & simpan model ke folder lokal
Jalankan **di komputer yang punya internet** (bukan di notebook ini), lalu salin foldernya ke komputer lomba, misalnya ke `../pretrained/indobert_base`:

```python
from transformers import AutoTokenizer, AutoModelForSequenceClassification

nama_model = "indobenchmark/indobert-base-p1"   # ganti sesuai pilihan
folder_tujuan = "indobert_base"                  # salin folder ini ke komputer lomba

AutoTokenizer.from_pretrained(nama_model).save_pretrained(folder_tujuan)
AutoModelForSequenceClassification.from_pretrained(nama_model, num_labels=2).save_pretrained(folder_tujuan)
```

## 1.3 Isi folder yang valid

| File | Fungsi | Wajib? |
|---|---|---|
| `config.json` | Arsitektur model | ✅ |
| `model.safetensors` **atau** `pytorch_model.bin` | Bobot model | ✅ |
| `vocab.txt` / `tokenizer.json` / `sentencepiece.bpe.model` | Kamus tokenizer | ✅ (salah satu) |
| `tokenizer_config.json`, `special_tokens_map.json` | Pengaturan tokenizer | disarankan |

## 1.4 Paket Python
Pastikan terpasang di komputer lomba (atau siapkan file `.whl` lewat `pip download`): `torch`, `transformers`, `scikit-learn`, `pandas`, `numpy`, `matplotlib`, `tqdm`, dan `sentencepiece` (jika memakai XLM-R).

> ⚠️ **Uji coba dulu secara offline** (matikan Wi-Fi) dengan `DEBUG = True` sebelum hari lomba.

# 2. Setup & konfigurasi

## 2.1 Import & mode offline
Dua variabel lingkungan berikut **harus diset sebelum** `import transformers`, supaya library tidak mencoba menghubungi internet:

| Variabel | Efek |
|---|---|
| `HF_HUB_OFFLINE=1` | Melarang akses ke Hugging Face Hub |
| `TRANSFORMERS_OFFLINE=1` | Melarang Transformers mengunduh apa pun |

In [ ]:
import os                                       # operasi path & variabel lingkungan

MODE_OFFLINE = True                             # True = larang semua akses internet (WAJIB True untuk lomba offline)
if MODE_OFFLINE:                                # kalau mode offline aktif
    os.environ["HF_HUB_OFFLINE"] = "1"          #   larang akses Hugging Face Hub
    os.environ["TRANSFORMERS_OFFLINE"] = "1"    #   larang unduhan dari Transformers
os.environ["TOKENIZERS_PARALLELISM"] = "false"  # hindari peringatan paralelisme tokenizer saat DataLoader dipakai

import re                                       # regular expression untuk membersihkan teks
import gc                                       # garbage collector (bebaskan memori)
import json                                     # menyimpan pemetaan label
import glob                                     # mencari file dengan pola nama
import html                                     # html.unescape: &amp; → &
import copy                                     # deepcopy bobot terbaik
import time                                     # mengukur durasi tiap epoch
import random                                   # keacakan Python
import contextlib                               # nullcontext untuk mode tanpa AMP

import numpy as np                              # array numerik
import pandas as pd                             # tabel data
import torch                                    # inti deep learning
import torch.nn as nn                           # nn.CrossEntropyLoss
from torch.utils.data import Dataset, DataLoader   # penyusun batch
from transformers import (
    AutoTokenizer,                              # tokenizer otomatis sesuai model
    AutoModelForSequenceClassification,         # model + head klasifikasi
    get_linear_schedule_with_warmup,            # jadwal learning rate
)
from sklearn.model_selection import train_test_split            # pembagi data
from sklearn.preprocessing import LabelEncoder                  # label teks ↔ angka
from sklearn.metrics import (accuracy_score, f1_score,          # metrik
                             classification_report, ConfusionMatrixDisplay)
from sklearn.feature_extraction.text import TfidfVectorizer     # fitur TF-IDF
from sklearn.pipeline import Pipeline, FeatureUnion             # menyusun tahapan model klasik
from sklearn.linear_model import LogisticRegression             # pengklasifikasi linear
import matplotlib.pyplot as plt                 # grafik
from tqdm.auto import tqdm                      # progress bar

import transformers                             # untuk mencetak versi
print("Torch:", torch.__version__, "| Transformers:", transformers.__version__, "| Offline:", MODE_OFFLINE)   # info versi

## 2.2 Konfigurasi — **bagian yang paling sering kamu ubah**

> 💡 Pada hari lomba, biasanya cukup mengubah blok **Data**, **Submission**, **Metrik**, dan **Model**.

| Blok | Variabel | Arti |
|---|---|---|
| Data | `FOLDER_DATA`, `FILE_TRAIN`, `FILE_TEST` | Lokasi pencarian & nama file |
| Data | `KOLOM_TEKS` | Nama kolom teks. Boleh **daftar** bila teks tersebar di beberapa kolom, mis. `["judul", "isi"]` |
| Data | `KOLOM_LABEL`, `KOLOM_ID` | Kolom label & ID |
| Submission | `KOLOM_ID_SUB`, `KOLOM_LABEL_SUB` | Nama kolom pada file yang dikirim (ikuti contoh `sample_submission.csv` dari panitia) |
| Metrik | `METRIK` | `"f1_makro"` (rata-rata F1 semua kelas), `"f1_biner"` (F1 kelas `1`), atau `"akurasi"` |
| Model | `FOLDER_MODEL` | Folder model lokal |
| Praproses | `TOKEN_URL`, `TOKEN_MENTION` | Pengganti URL/mention. Untuk IndoBERTweet: `"HTTPURL"` dan `"@USER"` |
| Training | `EPOCHS`, `BATCH_SIZE`, `LR`, `MAX_LEN` | Lihat tabel di bawah |

| Hyperparameter | Default | Penjelasan | Jika bermasalah |
|---|---|---|---|
| `MAX_LEN` | 128 | Batas jumlah token per teks | Teks panjang → naikkan (256); memori kurang / CPU → turunkan (64) |
| `BATCH_SIZE` | 16 | Teks per batch | `CUDA out of memory` → 8 |
| `EPOCHS` | 4 | Putaran atas data latih | Overfitting → kurangi; underfitting → tambah |
| `LR` | 3e-5 | Learning rate puncak | Training tidak stabil → 2e-5 atau 1e-5 |
| `KESABARAN` | 2 | Berhenti lebih awal jika skor validasi tidak naik selama sekian epoch | — |
| `PAKAI_BOBOT_KELAS` | `False` | Beri bobot lebih pada kelas langka | Aktifkan jika kelas **sangat timpang** |
| `LABEL_SMOOTHING` | 0.0 | Melunakkan target; berguna jika label berisik | Coba 0.1 |

In [ ]:
# ════════ BLOK DATA ════════
FOLDER_DATA = [".", "data", "../data", "input", "../input"]   # folder tempat mencari file (diperiksa berurutan)
FILE_TRAIN = "train.csv"                    # nama file data latih
FILE_TEST = "test.csv"                      # nama file data uji
PEMISAH_CSV = ","                           # pemisah kolom CSV (gunakan ";" atau "\t" bila perlu)
ENCODING_CSV = "utf-8"                      # encoding file (coba "latin-1" jika muncul error karakter)
KOLOM_TEKS = "text"                         # nama kolom teks; bisa daftar: ["judul", "isi"]
KOLOM_LABEL = "target"                      # nama kolom label
KOLOM_ID = "id"                             # nama kolom ID (jika tak ada di test, nomor baris dipakai)

# ════════ BLOK SUBMISSION ════════
FILE_SUBMISSION = "submission.csv"          # nama file hasil akhir
KOLOM_ID_SUB = KOLOM_ID                     # nama kolom ID di file submission
KOLOM_LABEL_SUB = KOLOM_LABEL               # nama kolom label di file submission

# ════════ BLOK METRIK ════════
METRIK = "f1_makro"                         # "f1_makro" | "f1_biner" | "akurasi"

# ════════ BLOK MODEL ════════
FOLDER_MODEL = "../pretrained/indobert_base"   # folder model lokal (format Hugging Face)
FOLDER_SIMPAN = "model_terbaik"             # tempat menyimpan model hasil fine-tuning

# ════════ BLOK PRAPROSES ════════
HURUF_KECIL = True                          # ubah ke huruf kecil (cocok untuk model uncased; set False untuk model cased)
NORMALISASI_SLANG = True                    # ganti singkatan/slang umum ("yg"→"yang"); matikan jika hasil memburuk
TOKEN_URL = "http"                          # pengganti URL (IndoBERTweet: "HTTPURL")
TOKEN_MENTION = "@user"                     # pengganti @mention (IndoBERTweet: "@USER")

# ════════ BLOK TRAINING ════════
SEED = 42                                   # seed keacakan
VAL_SPLIT = 0.2                             # porsi data validasi
MAX_LEN = 128                               # batas token per teks
BATCH_SIZE = 16                             # ukuran batch train
EPOCHS = 4                                  # jumlah epoch maksimum
LR = 3e-5                                   # learning rate puncak
WEIGHT_DECAY = 0.01                         # regularisasi L2
WARMUP_RATIO = 0.1                          # porsi langkah warmup
KESABARAN = 2                               # early stopping: toleransi epoch tanpa perbaikan
PAKAI_BOBOT_KELAS = False                   # bobot kelas untuk data timpang
LABEL_SMOOTHING = 0.0                       # label smoothing (0 = mati)

DEBUG = False                               # True = versi mini (300 baris, 1 epoch) untuk menguji pipeline
if DEBUG:                                   # mode uji cepat
    EPOCHS = 1                              #   cukup 1 epoch
    print("⚠️ DEBUG aktif: hasil bukan untuk disubmit.")   # peringatan


def set_seed(seed):                         # kunci semua sumber keacakan
    random.seed(seed)                       #   Python
    np.random.seed(seed)                    #   NumPy
    torch.manual_seed(seed)                 #   PyTorch CPU
    torch.cuda.manual_seed_all(seed)        #   PyTorch GPU (aman walau tanpa GPU)

set_seed(SEED)                              # terapkan seed

if torch.cuda.is_available():               # ada GPU NVIDIA?
    device = torch.device("cuda")           #   pakai GPU
elif getattr(torch.backends, "mps", None) and torch.backends.mps.is_available():   # GPU Apple?
    device = torch.device("mps")            #   pakai MPS
else:                                       # selain itu
    device = torch.device("cpu")            #   CPU (lambat untuk fine-tuning)
use_amp = device.type == "cuda"             # fp16 hanya di GPU NVIDIA
print("Device:", device, "| AMP:", use_amp) # tampilkan pilihan

# 3. Memuat data

`find_file` mencari file di setiap folder pada `FOLDER_DATA`; bila belum ketemu, mencari **rekursif** ke semua subfolder. Jadi struktur folder panitia yang tidak biasa pun biasanya tetap terbaca.

Setelah dimuat, notebook memeriksa:
- apakah kolom yang dikonfigurasi benar-benar ada (kalau tidak, pesan error menampilkan **daftar kolom yang tersedia**),
- sebaran label,
- panjang teks,
- teks duplikat (terutama yang **labelnya bertentangan**).

> ⚠️ Jika muncul `KeyError`/`ValueError` soal kolom, cukup sesuaikan `KOLOM_TEKS`, `KOLOM_LABEL`, `KOLOM_ID` pada bagian konfigurasi.

In [ ]:
def find_file(filename):                                      # cari file di folder kandidat, lalu rekursif
    for folder in FOLDER_DATA:                                #   periksa tiap folder kandidat
        path = os.path.join(folder, filename)                 #     bentuk path lengkap
        if os.path.exists(path):                              #     ada?
            return path                                       #       kembalikan
    hasil = glob.glob(os.path.join("**", filename), recursive=True)   # fallback: cari di semua subfolder
    if hasil:                                                 #   ketemu?
        return hasil[0]                                       #     pakai yang pertama
    raise FileNotFoundError(f"'{filename}' tidak ditemukan. Folder yang dicek: {FOLDER_DATA}")   # pesan jelas bila gagal


def baca_csv(filename):                                       # baca CSV dengan pengaturan dari konfigurasi
    path = find_file(filename)                                #   cari lokasinya
    print("Membaca:", path)                                   #   beri tahu lokasi yang dipakai
    return pd.read_csv(path, sep=PEMISAH_CSV, encoding=ENCODING_CSV)   # baca file


kolom_teks = [KOLOM_TEKS] if isinstance(KOLOM_TEKS, str) else list(KOLOM_TEKS)   # samakan menjadi daftar kolom

df_train = baca_csv(FILE_TRAIN)                               # data latih
df_test = baca_csv(FILE_TEST)                                 # data uji

for nama, df, wajib in [("train", df_train, kolom_teks + [KOLOM_LABEL]), ("test", df_test, kolom_teks)]:   # validasi kolom
    hilang = [k for k in wajib if k not in df.columns]        #   kolom yang diminta tetapi tidak ada
    if hilang:                                                #   ada yang hilang?
        raise ValueError(f"Kolom {hilang} tidak ada di data {nama}. Kolom tersedia: {list(df.columns)}")   # tampilkan kolom yang ada

if DEBUG:                                                     # mode uji cepat → kecilkan data
    df_train = df_train.sample(min(300, len(df_train)), random_state=0).reset_index(drop=True)   # 300 baris acak
    df_test = df_test.head(100).reset_index(drop=True)        #   100 baris pertama

df_train = df_train.dropna(subset=[KOLOM_LABEL]).reset_index(drop=True)   # buang baris tanpa label
print(f"Train: {df_train.shape} | Test: {df_test.shape}")     # ukuran data
df_train.head()                                               # intip data latih

In [ ]:
print("Kolom train:", list(df_train.columns))                 # semua kolom train
print("Kolom test :", list(df_test.columns))                  # semua kolom test
print("\nSebaran label (jumlah & persen):")                   # judul
dist = df_train[KOLOM_LABEL].value_counts()                   # hitung per kelas
print(pd.DataFrame({"jumlah": dist, "persen": (dist / dist.sum() * 100).round(1)}))   # tabel sebaran

teks_mentah = df_train[kolom_teks].fillna("").astype(str).agg(" | ".join, axis=1)   # teks gabungan (untuk statistik)
print("\nPanjang teks (jumlah kata):")                        # judul
print(teks_mentah.str.split().str.len().describe().round(1)) # statistik jumlah kata

n_dup = teks_mentah.duplicated().sum()                        # jumlah baris teks duplikat
g = df_train.groupby(teks_mentah)[KOLOM_LABEL].nunique()      # jumlah label berbeda per teks yang sama
n_konflik = int((g > 1).sum())                                # teks identik berlabel berbeda
print(f"\nTeks duplikat: {n_dup} | teks identik dengan label BERBEDA: {n_konflik}")   # laporan
if n_konflik:                                                 # ada konflik?
    print("⚠️ Ada label yang bertentangan; lihat ide peningkatan (bagian 11) untuk cara menanganinya.")   # peringatan

# 4. Praproses teks Indonesia & pembagian data

## 4.1 Normalisasi teks
Model Transformer sudah tahan terhadap teks mentah, jadi pembersihan dibuat **minimal**:

| Langkah | Contoh | Hasil | Alasan |
|---|---|---|---|
| Decode HTML | `&amp;` | `&` | Sisa encoding |
| URL | `https://t.co/xyz` | `TOKEN_URL` | Isi URL tak bermakna, keberadaannya bermakna |
| Mention | `@jokowi` | `TOKEN_MENTION` | Nama akun berisik |
| Huruf berulang | `bangeeeet`, `mantaaap` | `banget`, `mantap` | Pemanjangan huruf khas bahasa informal |
| Slang umum *(opsional)* | `yg`, `gak`, `tdk`, `krn` | `yang`, `tidak`, `tidak`, `karena` | Menyatukan ragam penulisan |
| Spasi berlebih | `a   b` | `a b` | Kerapian |

> 🧠 **Huruf berulang** hanya dirapikan untuk **huruf** yang muncul ≥ 3 kali berturut-turut (angka tidak disentuh, jadi `1000` tetap `1000`).

> ⚠️ **Kamus slang itu heuristik dan sengaja kecil.** Tambahkan istilah yang sering muncul di data lombamu (`KAMUS_SLANG["bgt"] = "banget"`). Untuk **IndoBERTweet**, coba matikan `NORMALISASI_SLANG` karena modelnya sudah melihat banyak teks informal.

## 4.2 Pembagian data
Data dibagi menjadi latih/validasi dengan **stratifikasi** (proporsi kelas dijaga). Jika ada kelas yang terlalu sedikit untuk distratifikasi, pembagian otomatis jatuh ke acak biasa.

In [ ]:
KAMUS_SLANG = {                                               # singkatan/slang → bentuk baku (tambahkan sendiri sesuai data)
    "yg": "yang", "dgn": "dengan", "utk": "untuk", "krn": "karena", "karna": "karena",   # kata hubung umum
    "tdk": "tidak", "gak": "tidak", "ga": "tidak", "gk": "tidak", "nggak": "tidak", "ngga": "tidak",   # kata negasi
    "blm": "belum", "udah": "sudah", "udh": "sudah", "sdh": "sudah",    # keterangan waktu
    "jgn": "jangan", "tp": "tapi", "klo": "kalau", "kalo": "kalau",     # kata sambung
    "aja": "saja", "bgt": "banget", "sy": "saya", "gw": "saya", "gue": "saya", "lu": "kamu",   # kata ganti & penegas
    "sbg": "sebagai", "dlm": "dalam", "pd": "pada", "org": "orang",     # lain-lain
}


def normalisasi_teks(teks):                                   # fungsi pembersih teks minimal
    teks = html.unescape(str(teks))                           #   decode entitas HTML; str() menjaga NaN tidak error
    if HURUF_KECIL:                                           #   opsi huruf kecil (dilakukan DULU agar token pengganti tidak ikut berubah)
        teks = teks.lower()                                   #     seluruh teks → huruf kecil
    teks = re.sub(r"https?://\S+|www\.\S+", f" {TOKEN_URL} ", teks)   # semua URL → token pengganti
    teks = re.sub(r"@\w+", TOKEN_MENTION, teks)               #   semua @mention → token pengganti
    teks = re.sub(r"([^\W\d_])\1{2,}", r"\1", teks)           #   huruf berulang ≥3 kali → 1 huruf (bangeeet → banget)
    if NORMALISASI_SLANG:                                     #   opsi normalisasi slang
        teks = re.sub(r"\b\w+\b", lambda m: KAMUS_SLANG.get(m.group(0).lower(), m.group(0)), teks)   # ganti kata yang ada di kamus
    teks = re.sub(r"\s+", " ", teks).strip()                  #   rapikan spasi
    return teks                                               #   hasil bersih


def gabung_teks(df):                                          # gabungkan beberapa kolom teks menjadi satu string per baris
    if len(kolom_teks) == 1:                                  #   hanya satu kolom?
        return df[kolom_teks[0]].fillna("").astype(str)       #     langsung pakai (NaN → "")
    return df[kolom_teks].fillna("").astype(str).agg(" | ".join, axis=1)   # beberapa kolom → disambung dengan " | "


# --- Contoh sebelum/sesudah ---
for t in gabung_teks(df_train).head(3):                       # tiga teks pertama
    print("ASLI :", t)                                        #   sebelum
    print("BERSIH:", normalisasi_teks(t), "\n")               #   sesudah

In [ ]:
texts_all = gabung_teks(df_train).map(normalisasi_teks).tolist()   # seluruh teks latih yang sudah dibersihkan
X_test = gabung_teks(df_test).map(normalisasi_teks).tolist()       # seluruh teks uji yang sudah dibersihkan
ids_test = df_test[KOLOM_ID].to_numpy() if KOLOM_ID in df_test.columns else np.arange(len(df_test))   # ID uji (atau nomor baris)

le = LabelEncoder()                                           # pengubah label ↔ angka 0..K-1
y_all = le.fit_transform(df_train[KOLOM_LABEL])               # ubah label asli menjadi angka
NUM_LABELS = len(le.classes_)                                 # jumlah kelas
print("Kelas :", list(le.classes_), "→", list(range(NUM_LABELS)))   # tampilkan pemetaan
if METRIK == "f1_biner" and NUM_LABELS != 2:                  # f1_biner hanya untuk 2 kelas
    raise ValueError("METRIK 'f1_biner' hanya untuk 2 kelas; gunakan 'f1_makro'.")   # hentikan dengan pesan jelas

with open("label_map.json", "w", encoding="utf-8") as f:      # simpan pemetaan label (berguna saat memuat model nanti)
    json.dump({str(i): str(c) for i, c in enumerate(le.classes_)}, f, ensure_ascii=False, indent=2)


def hitung_skor(y_benar, y_tebak):                            # satu fungsi untuk semua pilihan metrik
    if METRIK == "akurasi":                                   #   akurasi
        return accuracy_score(y_benar, y_tebak)               #     proporsi tebakan benar
    if METRIK == "f1_biner":                                  #   F1 kelas positif (indeks 1)
        return f1_score(y_benar, y_tebak)                     #     untuk 2 kelas
    if METRIK == "f1_makro":                                  #   rata-rata F1 semua kelas
        return f1_score(y_benar, y_tebak, average="macro")    #     adil untuk kelas timpang
    raise ValueError(f"METRIK tidak dikenal: {METRIK}")       #   salah ketik

try:                                                          # coba pembagian berstrata
    idx_tr, idx_va = train_test_split(np.arange(len(y_all)), test_size=VAL_SPLIT,
                                      random_state=SEED, stratify=y_all)
except ValueError:                                            # gagal (ada kelas dengan <2 contoh)
    print("⚠️ Stratifikasi gagal, memakai pembagian acak biasa.")   # beri tahu
    idx_tr, idx_va = train_test_split(np.arange(len(y_all)), test_size=VAL_SPLIT, random_state=SEED)

X_train = [texts_all[i] for i in idx_tr]                      # teks latih
X_val = [texts_all[i] for i in idx_va]                        # teks validasi
y_train, y_val = y_all[idx_tr], y_all[idx_va]                 # label latih & validasi
print(f"Train: {len(X_train)} | Val: {len(X_val)} | Test: {len(X_test)}")   # ukuran tiap bagian

# 5. Baseline 1: TF-IDF + Regresi Logistik

Model klasik ini tidak butuh GPU maupun model pra-latih.

## Cara kerja
**TF-IDF** memberi bobot tinggi pada kata yang **sering muncul di satu dokumen tetapi jarang di dokumen lain**:

$$\text{tfidf}(t,d)=\text{tf}(t,d)\times\log\frac{N}{\text{df}(t)}$$

Dua jenis fitur digabung:

| Fitur | Pengaturan | Gunanya |
|---|---|---|
| **Kata** (unigram + bigram) | `ngram_range=(1,2)` | Menangkap kata dan frasa pendek ("tidak suka") |
| **Karakter** (2–5 huruf) | `analyzer="char_wb"` | Tahan terhadap salah ketik, singkatan, dan imbuhan bahasa Indonesia (me-, -kan, -nya) |

Pengklasifikasinya **Regresi Logistik**, linear, cepat, dan sangat sulit dikalahkan untuk baseline teks.

> 💡 **Fungsi baseline ini ganda:** (1) jaring pengaman submission, (2) **patokan** — jika Transformer tidak jauh lebih baik dari angka ini, ada yang perlu dicurigai (data kecil, label berisik, atau konfigurasi kurang tepat).

In [ ]:
tfidf_model = Pipeline([                                      # rangkaian: fitur → pengklasifikasi
    ("fitur", FeatureUnion([                                  #   gabungkan dua jenis fitur
        ("kata", TfidfVectorizer(ngram_range=(1, 2),          #     fitur kata: unigram & bigram
                                 min_df=2,                    #       abaikan kata yang muncul <2 dokumen
                                 sublinear_tf=True,           #       redam dominasi kata yang sangat sering
                                 max_features=200000)),       #       batasi ukuran kamus
        ("char", TfidfVectorizer(analyzer="char_wb",          #     fitur karakter dalam batas kata
                                 ngram_range=(2, 5),          #       potongan 2–5 huruf
                                 min_df=2,                    #       abaikan yang sangat jarang
                                 sublinear_tf=True,           #       redam frekuensi besar
                                 max_features=300000)),       #       batasi ukuran kamus
    ])),
    ("clf", LogisticRegression(                               #   pengklasifikasi
        C=4.0,                                                #     kekuatan regularisasi (besar = lebih bebas)
        max_iter=1000,                                        #     batas iterasi optimasi
        class_weight="balanced" if PAKAI_BOBOT_KELAS else None,   #     bobot kelas bila diminta
    )),
])

t0 = time.time()                                              # catat waktu mulai
tfidf_model.fit(X_train, y_train)                             # latih pada data latih
pred_tfidf = tfidf_model.predict(X_val)                       # prediksi data validasi
skor_tfidf = hitung_skor(y_val, pred_tfidf)                   # hitung skor sesuai METRIK
print(f"TF-IDF + LogReg | {METRIK} validasi = {skor_tfidf:.4f} | akurasi = {accuracy_score(y_val, pred_tfidf):.4f} | {time.time()-t0:.1f} dtk")   # laporan

In [ ]:
def simpan_submission(pred_idx, nama_file):                   # fungsi: simpan prediksi (indeks kelas) sebagai file submission
    sub = pd.DataFrame({                                      #   susun tabel
        KOLOM_ID_SUB: ids_test,                               #     kolom ID
        KOLOM_LABEL_SUB: le.inverse_transform(pred_idx),      #     kolom label (angka → label asli)
    })
    sub.to_csv(nama_file, index=False)                        #   simpan tanpa kolom indeks pandas
    print(f"✔ {nama_file} tersimpan: {sub.shape}")            #   konfirmasi
    return sub                                                #   kembalikan tabelnya


pred_test_tfidf = tfidf_model.predict(X_test)                 # prediksi seluruh data uji
sub_tfidf = simpan_submission(pred_test_tfidf, "submission_tfidf.csv")   # simpan sebagai jaring pengaman
sub_tfidf.head()                                              # intip hasil

> ✅ **Sampai di sini kamu sudah punya `submission_tfidf.csv`.** Jika waktu lomba mepet atau model Transformer bermasalah, kirim file itu.

# 6. Baseline 2: Memuat model Transformer (offline)

## Apa yang terjadi saat fine-tuning?

```text
teks ─► Tokenizer ─► [CLS] token token ... [SEP] ─► IndoBERT (12 layer) ─► vektor [CLS] ─► head linear ─► K logit
                                                    └─ bobot pra-latih (sudah paham bahasa Indonesia) ─┘   └─ baru (dilatih) ─┘
```

Model pra-latih sudah "paham" bahasa Indonesia; fine-tuning hanya menyesuaikannya ke tugasmu.

> ⚠️ **Peringatan "Some weights … newly initialized" itu normal**: head klasifikasi memang baru dibuat acak, dan itulah yang akan dilatih.

## Pemeriksaan folder model
`cek_folder_model` memeriksa isi folder **sebelum** memuat, sehingga kesalahan seperti folder salah path, bobot hilang, atau format Keras terdeteksi dengan pesan yang jelas, bukan error panjang yang membingungkan.

> ⚠️ Karena `local_files_only=True`, jika ada file yang kurang, proses **berhenti** (tidak akan diam-diam mengunduh).

In [ ]:
def cek_folder_model(folder):                                 # periksa kelengkapan folder model → (valid?, daftar masalah)
    if not os.path.isdir(folder):                             #   folder tidak ada?
        return False, [f"folder '{folder}' tidak ditemukan (cek path relatif terhadap lokasi notebook)"]
    isi = os.listdir(folder)                                  #   daftar file di dalam folder
    masalah = []                                              #   wadah daftar masalah
    if "config.json" not in isi:                              #   file arsitektur
        masalah.append("config.json tidak ada")
    if not any(f in isi for f in ["model.safetensors", "pytorch_model.bin",
                                  "model.safetensors.index.json", "pytorch_model.bin.index.json"]):   # file bobot
        masalah.append("file bobot (model.safetensors / pytorch_model.bin) tidak ada")
    if not any(f in isi for f in ["vocab.txt", "tokenizer.json", "sentencepiece.bpe.model",
                                  "spiece.model", "vocab.json"]):   # file kamus tokenizer
        masalah.append("file tokenizer (vocab.txt / tokenizer.json / sentencepiece.bpe.model) tidak ada")
    if "model.weights.h5" in isi:                             #   ciri preset Keras
        masalah.append("ini preset Keras (model.weights.h5); PyTorch butuh format Hugging Face")
    return len(masalah) == 0, masalah                         #   valid jika tak ada masalah


valid, masalah = cek_folder_model(FOLDER_MODEL)               # periksa folder model yang dikonfigurasi
if not valid:                                                 # ada masalah?
    raise FileNotFoundError(                                  #   hentikan dengan pesan lengkap
        f"Folder model '{FOLDER_MODEL}' bermasalah:\n  - " + "\n  - ".join(masalah) +
        "\n\nLihat bagian 1 untuk cara menyiapkan model. (submission_tfidf.csv sudah tersedia sebagai cadangan.)")

tokenizer = AutoTokenizer.from_pretrained(FOLDER_MODEL, local_files_only=True)   # muat tokenizer dari folder lokal
model = AutoModelForSequenceClassification.from_pretrained(   # muat model + head klasifikasi
    FOLDER_MODEL,                                             #   folder lokal
    num_labels=NUM_LABELS,                                    #   jumlah kelas sesuai data
    local_files_only=True,                                    #   larang akses internet
)
model.to(device)                                              # pindahkan model ke GPU/CPU

n_params = sum(p.numel() for p in model.parameters())         # total parameter
print(f"Model siap: {model.config.model_type} | parameter: {n_params/1e6:.1f} M | jumlah kelas: {NUM_LABELS}")   # ringkasan

## Mengintip tokenisasi
Tokenizer memecah teks menjadi **subword**. Kata Indonesia berimbuhan sering terpecah menjadi akar + imbuhan atau potongan kecil. Ini normal.

Cek juga **berapa teks yang terpotong** oleh `MAX_LEN`: jika banyak (> ±5%), pertimbangkan menaikkan `MAX_LEN`.

In [ ]:
contoh = X_train[0]                                           # satu teks sebagai contoh
print("Teks      :", contoh)                                 # teks bersih
print("Token     :", tokenizer.tokenize(contoh))             # hasil pemecahan subword
enc_contoh = tokenizer(contoh, truncation=True, max_length=MAX_LEN)   # tokenisasi lengkap
print("input_ids :", enc_contoh["input_ids"])                 # ID angka yang masuk ke model

sampel = X_train[:2000]                                       # ambil sampel agar cepat
panjang = [len(x) for x in tokenizer(sampel, truncation=False)["input_ids"]]   # panjang token tanpa pemotongan
persen_terpotong = np.mean(np.array(panjang) > MAX_LEN) * 100 # persentase yang melebihi MAX_LEN
print(f"\nPanjang token: median {np.median(panjang):.0f} | persentil-95 {np.percentile(panjang, 95):.0f} | maks {max(panjang)}")   # statistik
print(f"Teks yang terpotong oleh MAX_LEN={MAX_LEN}: {persen_terpotong:.1f}%")   # laporan pemotongan

# 7. DataLoader

| Komponen | Tugas |
|---|---|
| `Dataset` | Mendefinisikan cara mengambil **satu** contoh |
| `DataLoader` | Menyusun banyak contoh menjadi **batch** (dan mengacak urutan) |
| `collate` | Memberi **padding dinamis**: tiap batch hanya dipad sampai teks terpanjang **di batch itu** |

Seluruh teks **ditokenisasi sekali di awal**, sehingga tidak diulang di setiap epoch.

> 🧠 Bentuk tensor per batch: `input_ids` dan `attention_mask` berukuran **(B, L)** — *B* jumlah teks di batch, *L* panjang token terpanjang di batch itu. `pad_to_multiple_of=8` membulatkan *L* ke kelipatan 8 agar GPU lebih efisien.

In [ ]:
def tokenisasi(teks_list):                                    # tokenisasi seluruh daftar teks sekali jalan
    enc = tokenizer(teks_list, truncation=True, max_length=MAX_LEN)   # potong jika > MAX_LEN
    return {k: v for k, v in enc.items()}                     #   dict of list (input_ids, attention_mask, ...)

enc_train = tokenisasi(X_train)                               # token data latih
enc_val = tokenisasi(X_val)                                   # token data validasi
enc_test = tokenisasi(X_test)                                 # token data uji


class DatasetTeks(Dataset):                                   # Dataset berbasis hasil tokenisasi
    def __init__(self, enc, labels=None):                     #   enc: dict of list ; labels: array/None
        self.enc = enc                                        #     hasil tokenisasi
        self.labels = labels                                  #     label (None untuk test)

    def __len__(self):                                        #   jumlah contoh
        return len(self.enc["input_ids"])                     #     = banyaknya teks

    def __getitem__(self, i):                                 #   ambil contoh ke-i
        item = {k: v[i] for k, v in self.enc.items()}         #     ambil semua field tokenisasi baris i
        if self.labels is not None:                           #     bila berlabel
            item["labels"] = int(self.labels[i])              #       tambahkan label
        return item                                           #     satu contoh


def collate(batch):                                           # gabungkan list contoh → 1 batch tensor
    labels = [f.pop("labels") for f in batch if "labels" in f]   # keluarkan label (tokenizer.pad tidak mengurus label)
    out = tokenizer.pad(batch, padding=True, pad_to_multiple_of=8, return_tensors="pt")   # padding dinamis → tensor
    if labels:                                                #   bila ada label
        out["labels"] = torch.tensor(labels, dtype=torch.long)   # tambahkan sebagai tensor int64
    return out                                                #   batch siap pakai


def buat_loader(enc, labels, shuffle, batch_size):            # pabrik DataLoader
    return DataLoader(DatasetTeks(enc, labels), batch_size=batch_size,   # dataset & ukuran batch
                      shuffle=shuffle, collate_fn=collate)    #   acak urutan? ; fungsi penyusun batch


train_loader = buat_loader(enc_train, y_train, True, BATCH_SIZE)          # loader latih (diacak)
val_loader = buat_loader(enc_val, y_val, False, BATCH_SIZE * 2)           # loader validasi
test_loader = buat_loader(enc_test, None, False, BATCH_SIZE * 2)          # loader uji (urutan WAJIB tetap)

contoh_batch = next(iter(train_loader))                       # ambil satu batch untuk dicek
print({k: tuple(v.shape) for k, v in contoh_batch.items()})   # bentuk tiap tensor

# 8. Training

## 8.1 Satu langkah training
```text
1. Forward   : batch → model → logits (B, K)
2. Loss      : bandingkan logits dengan label (cross-entropy)
3. Backward  : hitung gradien
4. Clip      : potong gradien besar (stabilitas)
5. Step      : optimizer memperbarui bobot
6. Scheduler : geser learning rate
```

## 8.2 Rumus
**Cross-entropy** (dengan bobot kelas opsional $w_c$ dan label smoothing $\varepsilon$):

$$\mathcal{L}=-\frac{1}{B}\sum_{i=1}^{B} w_{y_i}\,\log p_\theta(y_i\mid x_i),\qquad p=\mathrm{softmax}(\text{logits})$$

**Bobot kelas** (hanya bila `PAKAI_BOBOT_KELAS=True`), untuk $N$ data dan $K$ kelas: $w_c=\dfrac{N}{K\cdot n_c}$ → kelas langka mendapat bobot lebih besar.

**Jadwal learning rate** — naik linear selama warmup ($T_w$ langkah) lalu turun linear ke 0 pada langkah $T$:

$$\eta_t=\begin{cases}\eta_{\max}\,t/T_w & t<T_w\\ \eta_{\max}\,(T-t)/(T-T_w) & t\ge T_w\end{cases}$$

## 8.3 Early stopping & bobot terbaik
Setiap epoch dinilai di data validasi. Bobot dengan **skor validasi terbaik** disimpan, dan training berhenti bila skor tidak membaik selama `KESABARAN` epoch.

> 💡 **Cara membaca log:** `loss latih` terus turun tetapi `loss val` naik → *overfitting*. Skor val naik pelan lalu datar → wajar, biarkan early stopping bekerja.

<details>
<summary><b>▸ Detail: <code>GradScaler</code> & <code>unscale_</code></b></summary>

Di float16, gradien yang sangat kecil bisa menjadi 0 (*underflow*). `GradScaler` mengalikan loss dengan faktor besar sebelum `backward()`. Sebelum *gradient clipping*, gradien dikembalikan ke skala asli lewat `unscale_`. Jika AMP mati (CPU/MPS), semuanya otomatis tidak melakukan apa-apa.
</details>

In [ ]:
def to_device(batch):                                         # pindahkan semua tensor dalam batch ke device
    return {k: v.to(device) for k, v in batch.items()}        #   dict comprehension

def autocast_ctx():                                           # konteks presisi campuran
    if use_amp:                                               #   GPU NVIDIA?
        return torch.autocast(device_type="cuda", dtype=torch.float16)   # forward dalam fp16
    return contextlib.nullcontext()                           #   selain itu presisi normal


def bangun_optimizer(model):                                  # AdamW dengan dua grup parameter
    tanpa_decay = ["bias", "LayerNorm.weight", "layer_norm.weight"]   # parameter yang tak diberi weight decay
    grup = [
        {"params": [p for n, p in model.named_parameters() if not any(k in n for k in tanpa_decay)],
         "weight_decay": WEIGHT_DECAY},                       #   grup 1: bobot biasa (kena decay)
        {"params": [p for n, p in model.named_parameters() if any(k in n for k in tanpa_decay)],
         "weight_decay": 0.0},                                #   grup 2: bias & LayerNorm (tanpa decay)
    ]
    return torch.optim.AdamW(grup, lr=LR)                     #   optimizer


optimizer = bangun_optimizer(model)                           # buat optimizer
total_langkah = len(train_loader) * EPOCHS                    # total langkah training
scheduler = get_linear_schedule_with_warmup(                  # jadwal learning rate
    optimizer, int(WARMUP_RATIO * total_langkah), total_langkah)   # warmup lalu turun linear
scaler = torch.amp.GradScaler("cuda", enabled=use_amp)        # skala loss fp16 (mati bila bukan GPU NVIDIA)

if PAKAI_BOBOT_KELAS:                                         # bobot kelas aktif?
    jumlah = np.maximum(np.bincount(y_train, minlength=NUM_LABELS), 1)   # jumlah contoh per kelas (min 1 agar tak bagi nol)
    bobot_kelas = torch.tensor(len(y_train) / (NUM_LABELS * jumlah), dtype=torch.float).to(device)   # w_c = N / (K·n_c)
    print("Bobot kelas:", bobot_kelas.cpu().numpy().round(2))  #   tampilkan
else:                                                         # tidak aktif
    bobot_kelas = None                                        #   tanpa bobot
criterion = nn.CrossEntropyLoss(weight=bobot_kelas, label_smoothing=LABEL_SMOOTHING)   # fungsi loss


def latih_satu_epoch(epoch):                                  # latih model selama satu epoch
    model.train()                                             #   mode train (dropout aktif)
    total_loss, benar, dilihat = 0.0, 0, 0                    #   akumulator
    pbar = tqdm(train_loader, desc=f"Epoch {epoch}/{EPOCHS} [latih]")   # progress bar
    for batch in pbar:                                        #   tiap batch
        labels = batch.pop("labels").to(device)               #     pisahkan label (loss dihitung manual)
        batch = to_device(batch)                              #     kirim input ke device
        optimizer.zero_grad(set_to_none=True)                 #     nolkan gradien lama
        with autocast_ctx():                                  #     forward dalam presisi campuran
            logits = model(**batch).logits                    #       logits (B, K)
        loss = criterion(logits.float(), labels)              #     loss dalam fp32 agar stabil
        scaler.scale(loss).backward()                         #     backward dengan loss berskala
        scaler.unscale_(optimizer)                            #     kembalikan skala gradien sebelum clipping
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)   #   potong norma gradien ≤ 1.0
        scaler.step(optimizer)                                #     update bobot
        scaler.update()                                       #     perbarui faktor skala
        scheduler.step()                                      #     geser learning rate
        bs = labels.size(0)                                   #     jumlah contoh di batch
        total_loss += loss.item() * bs                        #     akumulasi loss
        benar += (logits.argmax(-1) == labels).sum().item()   #     akumulasi tebakan benar
        dilihat += bs                                         #     akumulasi jumlah contoh
        pbar.set_postfix(loss=total_loss / dilihat, acc=benar / dilihat)   # tampilkan di progress bar
    return total_loss / dilihat, benar / dilihat              #   rata-rata loss & akurasi epoch ini


@torch.no_grad()                                              # tanpa gradien (inferensi)
def prediksi_probabilitas(loader):                            # keluaran: probabilitas (N, K)
    model.eval()                                              #   mode eval (dropout mati)
    hasil = []                                                #   wadah per batch
    for batch in loader:                                      #   tiap batch
        batch.pop("labels", None)                             #     buang label bila ada
        batch = to_device(batch)                              #     kirim ke device
        with autocast_ctx():                                  #     fp16 bila aktif
            logits = model(**batch).logits                    #       logits (B, K)
        hasil.append(torch.softmax(logits.float(), dim=-1).cpu())   # softmax → probabilitas → CPU
    return torch.cat(hasil).numpy()                           #   gabung semua batch


def evaluasi(loader, y_true):                                 # skor & loss pada loader berlabel
    probs = prediksi_probabilitas(loader)                     #   probabilitas tiap kelas
    pred = probs.argmax(axis=1)                               #   kelas dengan probabilitas tertinggi
    loss = -np.log(np.clip(probs[np.arange(len(y_true)), y_true], 1e-9, None)).mean()   # cross-entropy sederhana
    return hitung_skor(y_true, pred), loss, probs             #   skor, loss, probabilitas

In [ ]:
riwayat = {"loss_latih": [], "loss_val": [], "skor_val": []}  # catatan metrik per epoch
skor_terbaik, state_terbaik, tanpa_perbaikan = -1.0, None, 0  # rekor terbaik & penghitung kesabaran
t_mulai = time.time()                                         # waktu mulai training

for epoch in range(1, EPOCHS + 1):                            # loop epoch
    t_ep = time.time()                                        #   waktu mulai epoch
    loss_latih, acc_latih = latih_satu_epoch(epoch)           #   latih satu epoch
    skor_val, loss_val, _ = evaluasi(val_loader, y_val)       #   nilai di data validasi

    riwayat["loss_latih"].append(loss_latih)                  #   catat loss latih
    riwayat["loss_val"].append(loss_val)                      #   catat loss validasi
    riwayat["skor_val"].append(skor_val)                      #   catat skor validasi
    print(f"Epoch {epoch}/{EPOCHS} | loss latih {loss_latih:.4f} | loss val {loss_val:.4f} | "
          f"{METRIK} val {skor_val:.4f} | {time.time()-t_ep:.0f} dtk")   # ringkasan per epoch

    if skor_val > skor_terbaik:                               #   skor validasi membaik?
        skor_terbaik = skor_val                               #     perbarui rekor
        state_terbaik = copy.deepcopy({k: v.detach().cpu() for k, v in model.state_dict().items()})   # salin bobot ke CPU
        tanpa_perbaikan = 0                                   #     reset penghitung
    else:                                                     #   tidak membaik
        tanpa_perbaikan += 1                                  #     tambah penghitung
        if tanpa_perbaikan >= KESABARAN:                      #     sudah melewati batas kesabaran?
            print(f"⏹ Early stopping: tak ada perbaikan selama {KESABARAN} epoch.")   # info
            break                                             #       hentikan training

model.load_state_dict(state_terbaik)                          # kembalikan bobot terbaik
model.to(device)                                              # pastikan di device
print(f"\nSelesai dalam {(time.time()-t_mulai)/60:.1f} menit | {METRIK} validasi terbaik = {skor_terbaik:.4f} "
      f"(TF-IDF: {skor_tfidf:.4f})")                          # bandingkan dengan baseline klasik

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(10, 3.5))               # dua grafik berdampingan
ax[0].plot(riwayat["loss_latih"], marker="o", label="latih")  # kurva loss latih
ax[0].plot(riwayat["loss_val"], marker="o", label="validasi") # kurva loss validasi
ax[0].set_title("Loss"); ax[0].set_xlabel("epoch"); ax[0].legend()   # judul & legenda
ax[1].plot(riwayat["skor_val"], marker="o", color="green")    # kurva skor validasi
ax[1].axhline(skor_tfidf, ls="--", c="gray", label="TF-IDF")  # garis patokan baseline klasik
ax[1].set_title(f"{METRIK} validasi"); ax[1].set_xlabel("epoch"); ax[1].legend()   # judul & legenda
plt.tight_layout(); plt.show()                                # rapikan & tampilkan

# 9. Evaluasi

## Metrik

Dari confusion matrix tiap kelas: **TP**, **FP**, **FN**.

$$P=\frac{TP}{TP+FP},\qquad R=\frac{TP}{TP+FN},\qquad F_1=\frac{2PR}{P+R}$$

| Metrik | Cara menghitung | Cocok untuk |
|---|---|---|
| `akurasi` | Proporsi tebakan benar | Kelas seimbang |
| `f1_biner` | $F_1$ kelas positif saja | 2 kelas, kelas `1` yang penting |
| `f1_makro` | Rata-rata $F_1$ semua kelas | Multikelas / kelas timpang |

> 🧠 **Laporan klasifikasi** menunjukkan *precision*, *recall*, dan $F_1$ **per kelas**, jadi kelas mana yang paling sering salah langsung terlihat. Kelas dengan recall rendah biasanya perlu bobot kelas atau data tambahan.

In [ ]:
skor_val, loss_val, probs_val = evaluasi(val_loader, y_val)   # evaluasi akhir dengan bobot terbaik
pred_val = probs_val.argmax(axis=1)                           # kelas prediksi
print(f"{METRIK} validasi (Transformer) = {skor_val:.4f}  |  TF-IDF = {skor_tfidf:.4f}\n")   # bandingkan dengan baseline klasik
print(classification_report(y_val, pred_val, target_names=[str(c) for c in le.classes_], digits=3, zero_division=0))   # laporan per kelas

In [ ]:
disp = ConfusionMatrixDisplay.from_predictions(               # gambar confusion matrix validasi
    y_val, pred_val,                                          #   label asli vs prediksi
    display_labels=[str(c) for c in le.classes_],             #   nama kelas asli di sumbu
    cmap=plt.cm.Blues, xticks_rotation=45)                    #   warna & rotasi label
disp.ax_.set_title(f"Confusion Matrix Validasi — {METRIK}: {skor_val:.3f}")   # judul
plt.show()                                                    # tampilkan

## Analisis kesalahan
Melihat teks yang **paling meleset** sering membuka pola: label meragukan, sarkasme, singkatan yang belum ada di kamus slang, dsb.

In [ ]:
analisis = pd.DataFrame({                                     # tabel analisis kesalahan pada data validasi
    "teks": X_val,                                            #   teks bersih
    "label_asli": le.inverse_transform(y_val),                #   label asli
    "label_tebak": le.inverse_transform(pred_val),            #   label prediksi
    "p_benar": probs_val[np.arange(len(y_val)), y_val],       #   probabilitas yang diberikan model pada label ASLI
})
pd.set_option("display.max_colwidth", 120)                    # tampilkan teks lebih panjang
analisis[analisis["label_asli"] != analisis["label_tebak"]].sort_values("p_benar").head(15).round(3)   # salah tebak dengan keyakinan terendah pada label asli

# 10. Prediksi & submission

Probabilitas data uji disimpan (`probs_test.npy`) supaya nanti bisa **dirata-rata dengan model lain** tanpa melatih ulang.

> ⚠️ **Urutan baris harus sama dengan `df_test`.** Itu sebabnya loader uji memakai `shuffle=False` dan ID diambil langsung dari `df_test`.
> ⚠️ Pastikan **nama kolom & format** sesuai `sample_submission.csv` dari panitia (atur `KOLOM_ID_SUB` dan `KOLOM_LABEL_SUB`).

In [ ]:
probs_test = prediksi_probabilitas(test_loader)               # probabilitas tiap kelas untuk seluruh data uji, bentuk (N, K)
np.save("probs_test.npy", probs_test)                         # simpan untuk ensemble nanti
np.save("probs_val.npy", probs_val)                           # simpan probabilitas validasi juga

pred_test = probs_test.argmax(axis=1)                         # kelas prediksi
sub = simpan_submission(pred_test, "submission_transformer.csv" if DEBUG else FILE_SUBMISSION)   # simpan file submission
print("\nSebaran prediksi:")                                  # judul
print(sub[KOLOM_LABEL_SUB].value_counts(normalize=True).round(3))   # proporsi tiap kelas (bandingkan dengan sebaran train)
sub.head()                                                    # intip hasil

## (Opsional) Simpan model hasil fine-tuning
Tersimpan dalam format Hugging Face dan bisa dimuat ulang offline dengan `AutoModelForSequenceClassification.from_pretrained(FOLDER_SIMPAN, local_files_only=True)`.

In [ ]:
model.save_pretrained(FOLDER_SIMPAN)                          # simpan bobot + config
tokenizer.save_pretrained(FOLDER_SIMPAN)                      # simpan tokenizer
print("Model tersimpan di:", FOLDER_SIMPAN)                   # konfirmasi

# 11. Ide peningkatan, checklist hari-H & pemecahan masalah

## 11.1 Ide peningkatan (urut dari yang paling murah)

| Ide | Cara | Catatan |
|---|---|---|
| **Ganti model** | Ubah `FOLDER_MODEL` (IndoBERT-p1 ↔ IndoBERTweet ↔ XLM-R) | Biasanya perbedaan terbesar; bandingkan di data validasi yang sama |
| **Ensemble TF-IDF + Transformer** | Rata-ratakan `probs` kedua model dengan bobot (mis. 0,3 : 0,7) | Murah, sering menambah skor sedikit |
| **Bobot kelas** | `PAKAI_BOBOT_KELAS = True` | Untuk data timpang, terutama dengan `f1_makro` |
| **Atur `MAX_LEN`** | Lihat persentase teks terpotong di bagian 6 | Teks panjang → 256 |
| **Perkaya kamus slang** | Tambah istilah yang sering muncul di data | Lihat kolom `teks` pada analisis kesalahan |
| **Rapikan label bertentangan** | Seragamkan teks duplikat ke label mayoritas, buang yang seri | Bagian 3 menunjukkan jumlahnya |
| **Cross-validation 5-fold + rata-rata prediksi** | Latih 5 model, rata-ratakan prediksi uji | Lebih stabil, tapi 5× lebih lama; pola kodenya ada di notebook `disaster-tweets-roberta-cv-pytorch.ipynb` |
| **Latih ulang di seluruh data** | Setelah konfigurasi terbaik ditemukan, latih dengan `VAL_SPLIT` sangat kecil, jumlah epoch = epoch terbaik | Memakai semua data, tapi tanpa validasi |
| **Optimasi ambang (2 kelas)** | Cari ambang F1 terbaik pada probabilitas validasi | Hati-hati overfitting ke validasi |

## 11.2 ✅ Checklist hari-H

- [ ] Folder model sudah disalin dan **terbukti terbaca saat offline** (`DEBUG = True` berjalan sampai selesai).
- [ ] Paket Python terpasang (`torch`, `transformers`, `scikit-learn`, `sentencepiece` bila perlu).
- [ ] `KOLOM_TEKS`, `KOLOM_LABEL`, `KOLOM_ID`, `METRIK` sesuai deskripsi lomba.
- [ ] Nama kolom submission sesuai `sample_submission.csv`.
- [ ] Jalankan sampai bagian 5 → **kirim `submission_tfidf.csv` lebih dulu** sebagai pengaman.
- [ ] Jalankan Transformer; bandingkan skor validasinya dengan TF-IDF.
- [ ] Cek jumlah baris submission = jumlah baris data uji, dan tidak ada ID ganda.
- [ ] Catat konfigurasi & skor setiap percobaan (mis. di sel markdown) agar tak bingung nanti.

## 11.3 Pemecahan masalah

| Gejala | Kemungkinan penyebab | Solusi |
|---|---|---|
| `FileNotFoundError: 'train.csv' tidak ditemukan` | Nama/lokasi file berbeda | Ubah `FILE_TRAIN`, `FILE_TEST`, `FOLDER_DATA` |
| `ValueError: Kolom [...] tidak ada` | Nama kolom berbeda | Lihat daftar kolom pada pesan error, ubah `KOLOM_*` |
| `UnicodeDecodeError` | Encoding bukan UTF-8 | `ENCODING_CSV = "latin-1"` |
| Semua kolom menyatu jadi satu | Pemisah CSV bukan koma | `PEMISAH_CSV = ";"` atau `"\t"` |
| `Folder model ... bermasalah` | Isi folder tidak lengkap / format Keras | Ikuti bagian 1 |
| Error terkait `sentencepiece` | XLM-R butuh paket ini | `pip install sentencepiece` (siapkan `.whl`-nya sebelum lomba) |
| `CUDA out of memory` | Batch/`MAX_LEN` terlalu besar | `BATCH_SIZE = 8`, `MAX_LEN = 64` |
| Training sangat lambat | Berjalan di CPU | Pakai model lebih kecil / `MAX_LEN` 64 / batasi `EPOCHS`; andalkan TF-IDF |
| Loss `nan` | fp16 tidak stabil | `use_amp = False`, atau turunkan `LR` |
| Skor Transformer ≈ TF-IDF atau lebih buruk | Data kecil / label berisik / `LR` kurang cocok | Coba `LR = 2e-5`, `EPOCHS = 5`, model lain, bobot kelas |
| Semua prediksi satu kelas | `LR` terlalu besar atau data timpang parah | `LR = 1e-5`, `PAKAI_BOBOT_KELAS = True` |